# 🚀 Driving Narrator - Full GPU Benchmark (T4)

This notebook evaluates the **maximum performance** of the Driving Narrator model on a T4 GPU.

## What We Measure:
1. **Accuracy:** mAP, Precision, Recall, Per-Class breakdown
2. **Static Speed:** FPS at 416, 640, 1280 resolutions
3. **Video Speed:** Real video FPS with frame-by-frame tracking
4. **Throughput:** Batch processing speed
5. **Visualizations:** Confusion matrix, PR curves, benchmark charts

---

## 1️⃣ Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python-headless matplotlib pandas seaborn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ========================================
# PATHS (matching your training notebook)
# ========================================
from pathlib import Path
import os
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/Driving_Narrator')

# Model path (from DN_Training_v3)
MODEL_PATH = DRIVE_BASE / 'DN_Training_v3' / 'checkpoints' / 'yolo11n_lisa' / 'weights' / 'best.pt'

# Dataset (will be copied to /content for speed)
DATASET_DRIVE = DRIVE_BASE / 'LISA_stratified'
DATASET_LOCAL = Path('/content/LISA_stratified')

# Video for testing (optional)
VIDEO_PATH = str(DRIVE_BASE / 'test_video.mp4')

# Reports output
REPORTS_DIR = str(DRIVE_BASE / 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# Copy dataset to local SSD for faster I/O
if DATASET_LOCAL.exists():
    print('Dataset already in /content')
else:
    # Try using ZIP first (Much faster)
    zip_path = DRIVE_BASE / 'LISA_stratified.zip'
    if zip_path.exists():
        print(f'🚀 Found zip at {zip_path}, copying and unzipping (Fast)...')
        shutil.copy(zip_path, '/content/LISA_stratified.zip')
        
        # Unzip
        print('Unzipping...')
        import zipfile
        with zipfile.ZipFile('/content/LISA_stratified.zip', 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print('✅ Done!')
    else:
        print('⚠️ Zip not found, copying folder (Slow)...')
        shutil.copytree(DATASET_DRIVE, DATASET_LOCAL)
        print('✅ Done!')

# Update data.yaml for Colab paths
data_yaml_content = '''train: /content/LISA_stratified/train/images
val: /content/LISA_stratified/valid/images
test: /content/LISA_stratified/test/images

nc: 47
names: ['addedLane', 'curveLeft', 'curveRight', 'dip', 'doNotEnter', 'doNotPass', 'intersection', 'keepRight', 'laneEnds', 'merge', 'noLeftTurn', 'noRightTurn', 'pedestrianCrossing', 'rampSpeedAdvisory20', 'rampSpeedAdvisory35', 'rampSpeedAdvisory40', 'rampSpeedAdvisory45', 'rampSpeedAdvisory50', 'rampSpeedAdvisoryUrdbl', 'rightLaneMustTurn', 'roundabout', 'school', 'schoolSpeedLimit25', 'signalAhead', 'slow', 'speedLimit15', 'speedLimit25', 'speedLimit30', 'speedLimit35', 'speedLimit40', 'speedLimit45', 'speedLimit50', 'speedLimit55', 'speedLimit65', 'speedLimitUrdbl', 'stop', 'stopAhead', 'thruMergeLeft', 'thruMergeRight', 'thruTrafficMergeLeft', 'truckSpeedLimit55', 'turnLeft', 'turnRight', 'yield', 'yieldAhead', 'zoneAhead25', 'zoneAhead45']
'''

DATA_YAML = '/content/LISA_stratified/data.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(data_yaml_content)

print(f"\n✅ Model exists: {MODEL_PATH.exists()}")
print(f"✅ Dataset ready: {DATASET_LOCAL.exists()}")
print(f"✅ Video exists: {os.path.exists(VIDEO_PATH)}")

## 2️⃣ Load Model

In [ ]:
from ultralytics import YOLO
import time
import numpy as np
import cv2
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load the PyTorch model
model = YOLO(str(MODEL_PATH))
print(f"✅ Model loaded: {MODEL_PATH}")
print(f"   Classes: {len(model.names)}")
print(f"   Names: {list(model.names.values())[:5]}...")

## 3️⃣ Accuracy Evaluation

Full validation with confusion matrix, PR curves, and per-class metrics.

In [ ]:
# Evaluate on VALIDATION set
print("=" * 60)
print("VALIDATION SET EVALUATION")
print("=" * 60)

val_results = model.val(
    data=DATA_YAML,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    save_json=True,
    verbose=True
)

print(f"\n📊 Validation Results:")
print(f"  mAP@0.5:      {val_results.box.map50*100:.2f}%")
print(f"  mAP@0.5:0.95: {val_results.box.map*100:.2f}%")
print(f"  Precision:    {val_results.box.mp*100:.2f}%")
print(f"  Recall:       {val_results.box.mr*100:.2f}%")

In [ ]:
# Evaluate on TEST set
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

test_results = model.val(
    data=DATA_YAML,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    save_json=True,
    verbose=True
)

print(f"\n📊 Test Results:")
print(f"  mAP@0.5:      {test_results.box.map50*100:.2f}%")
print(f"  mAP@0.5:0.95: {test_results.box.map*100:.2f}%")
print(f"  Precision:    {test_results.box.mp*100:.2f}%")
print(f"  Recall:       {test_results.box.mr*100:.2f}%")

In [ ]:
# Copy generated plots to reports folder
import glob

val_dirs = sorted(glob.glob('runs/detect/val*'), key=os.path.getmtime, reverse=True)
if val_dirs:
    latest_val = val_dirs[0]
    print(f"Copying plots from {latest_val}...")
    
    plots_to_copy = [
        'confusion_matrix.png',
        'confusion_matrix_normalized.png', 
        'PR_curve.png',
        'F1_curve.png',
    ]
    
    for plot in plots_to_copy:
        src = f'{latest_val}/{plot}'
        if os.path.exists(src):
            dst = f'{REPORTS_DIR}/gpu_{plot}'
            shutil.copy(src, dst)
            print(f"  ✅ Saved: {dst}")

print("\n📁 All plots saved to reports/ folder")

In [ ]:
# Per-class mAP breakdown
print("\n" + "=" * 60)
print("PER-CLASS mAP@0.5 (Test Set)")
print("=" * 60)

per_class_ap = test_results.box.ap50
class_names = list(model.names.values())

class_df = pd.DataFrame({
    'Class': class_names,
    'mAP@0.5': per_class_ap
}).sort_values('mAP@0.5', ascending=False)

print(class_df.to_string(index=False))

class_df.to_csv(f'{REPORTS_DIR}/per_class_map.csv', index=False)
print(f"\n✅ Saved: {REPORTS_DIR}/per_class_map.csv")

## 4️⃣ Static Speed Benchmarks

In [ ]:
def benchmark_static(model, imgsz, num_warmup=50, num_iterations=200):
    """Benchmark inference on static images."""
    dummy_img = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)
    
    # Warmup
    for _ in range(num_warmup):
        model(dummy_img, verbose=False)
    torch.cuda.synchronize()
    
    # Benchmark
    latencies = []
    for _ in range(num_iterations):
        start = time.perf_counter()
        model(dummy_img, verbose=False)
        torch.cuda.synchronize()
        latencies.append((time.perf_counter() - start) * 1000)
    
    return {
        'imgsz': imgsz,
        'avg_ms': round(np.mean(latencies), 2),
        'std_ms': round(np.std(latencies), 2),
        'fps': round(1000 / np.mean(latencies), 1),
        'min_ms': round(min(latencies), 2),
        'max_ms': round(max(latencies), 2),
    }

# Benchmark resolutions
print("=" * 60)
print("STATIC SPEED BENCHMARK")
print("=" * 60)

resolutions = [416, 640, 1280]
speed_results = []

for imgsz in resolutions:
    print(f"\n🔄 Testing {imgsz}x{imgsz}...")
    result = benchmark_static(model, imgsz)
    speed_results.append(result)
    print(f"  ✅ {result['fps']} FPS | {result['avg_ms']}ms ± {result['std_ms']}ms")

# Summary table
print("\n" + "=" * 50)
print(f"{'Resolution':<12} {'FPS':>10} {'Latency':>12} {'Std':>10}")
print("-" * 50)
for r in speed_results:
    print(f"{r['imgsz']}x{r['imgsz']:<7} {r['fps']:>10.1f} {r['avg_ms']:>10.2f}ms {r['std_ms']:>8.2f}ms")

## 5️⃣ Video Benchmark (Real-Time FPS Tracking)

Processes a real video file with frame-by-frame FPS tracking.

In [ ]:
def benchmark_video(model, video_path, imgsz=640, skip_seconds=5, max_frames=500):
    """
    Benchmark on real video with frame-by-frame FPS tracking.
    
    Args:
        skip_seconds: Skip first N seconds for warmup
        max_frames: Maximum frames to process
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open video: {video_path}")
        return None
    
    fps_video = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"📹 Video Info:")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps_video}")
    print(f"   Total Frames: {total_frames}")
    
    # Skip first N seconds
    skip_frames = int(skip_seconds * fps_video)
    cap.set(cv2.CAP_PROP_POS_FRAMES, skip_frames)
    print(f"   Skipping first {skip_seconds}s ({skip_frames} frames)...")
    
    # Warmup (process 30 frames without timing)
    print("   Warming up...")
    for _ in range(30):
        ret, frame = cap.read()
        if ret:
            model(frame, imgsz=imgsz, verbose=False)
    torch.cuda.synchronize()
    
    # Benchmark
    frame_latencies = []
    frame_fps = []
    detections_per_frame = []
    
    print(f"   Processing up to {max_frames} frames...")
    
    for i in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        start = time.perf_counter()
        results = model(frame, imgsz=imgsz, verbose=False)
        torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) * 1000
        
        frame_latencies.append(elapsed_ms)
        frame_fps.append(1000 / elapsed_ms)
        detections_per_frame.append(len(results[0].boxes))
        
        if (i + 1) % 100 == 0:
            print(f"      Frame {i+1}: {1000/elapsed_ms:.1f} FPS")
    
    cap.release()
    
    return {
        'video_resolution': f'{width}x{height}',
        'inference_resolution': imgsz,
        'frames_processed': len(frame_latencies),
        'avg_fps': round(np.mean(frame_fps), 1),
        'max_fps': round(max(frame_fps), 1),
        'min_fps': round(min(frame_fps), 1),
        'std_fps': round(np.std(frame_fps), 2),
        'avg_latency_ms': round(np.mean(frame_latencies), 2),
        'avg_detections': round(np.mean(detections_per_frame), 2),
        'frame_fps': frame_fps,
        'frame_latencies': frame_latencies,
    }

In [ ]:
# Run video benchmark
video_results = None

if os.path.exists(VIDEO_PATH):
    print("=" * 60)
    print("VIDEO BENCHMARK")
    print("=" * 60)
    
    video_results = benchmark_video(
        model, 
        VIDEO_PATH, 
        imgsz=640,
        skip_seconds=5,
        max_frames=500
    )
    
    if video_results:
        print(f"\n📊 Video Benchmark Results:")
        print(f"  Average FPS:  {video_results['avg_fps']}")
        print(f"  Max FPS:      {video_results['max_fps']}")
        print(f"  Min FPS:      {video_results['min_fps']}")
        print(f"  Std Dev:      {video_results['std_fps']}")
        print(f"  Avg Latency:  {video_results['avg_latency_ms']}ms")
        print(f"  Avg Detections/Frame: {video_results['avg_detections']}")
else:
    print(f"⚠️ Video not found: {VIDEO_PATH}")
    print("   Skipping video benchmark.")

In [ ]:
# Plot frame-by-frame FPS
if video_results and video_results['frame_fps']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # FPS over time
    ax1 = axes[0]
    ax1.plot(video_results['frame_fps'], alpha=0.7, linewidth=0.8)
    ax1.axhline(video_results['avg_fps'], color='red', linestyle='--', label=f"Avg: {video_results['avg_fps']} FPS")
    ax1.set_xlabel('Frame Number')
    ax1.set_ylabel('FPS')
    ax1.set_title('Frame-by-Frame FPS')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # FPS histogram
    ax2 = axes[1]
    ax2.hist(video_results['frame_fps'], bins=30, edgecolor='black', alpha=0.7)
    ax2.axvline(video_results['avg_fps'], color='red', linestyle='--', label=f"Avg: {video_results['avg_fps']}")
    ax2.set_xlabel('FPS')
    ax2.set_ylabel('Frequency')
    ax2.set_title('FPS Distribution')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(f'{REPORTS_DIR}/gpu_video_fps_analysis.png', dpi=150)
    plt.show()
    print(f"\n✅ Saved: {REPORTS_DIR}/gpu_video_fps_analysis.png")

## 6️⃣ Batch Throughput Test

In [ ]:
def benchmark_batch(model, imgsz, batch_size, num_iterations=50):
    """Benchmark batched inference throughput."""
    batch = [np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8) 
             for _ in range(batch_size)]
    
    # Warmup
    for _ in range(10):
        model(batch, verbose=False)
    torch.cuda.synchronize()
    
    # Benchmark
    start = time.perf_counter()
    for _ in range(num_iterations):
        model(batch, verbose=False)
        torch.cuda.synchronize()
    total_time = time.perf_counter() - start
    
    total_images = batch_size * num_iterations
    throughput = total_images / total_time
    
    return {
        'batch_size': batch_size,
        'throughput_ips': round(throughput, 1),
        'latency_per_batch_ms': round((total_time / num_iterations) * 1000, 2)
    }

# Test batch sizes
print("=" * 60)
print("BATCH THROUGHPUT BENCHMARK (640x640)")
print("=" * 60)

batch_sizes = [1, 2, 4, 8, 16, 32]
throughput_results = []

for bs in batch_sizes:
    try:
        print(f"\n🔄 Batch size = {bs}...")
        result = benchmark_batch(model, 640, bs)
        throughput_results.append(result)
        print(f"  ✅ {result['throughput_ips']} images/sec")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"  ⚠️ OOM at batch_size={bs}")
            torch.cuda.empty_cache()
            break
        raise

# Summary
print("\n" + "=" * 50)
print(f"{'Batch':<10} {'Throughput':>15} {'Batch Latency':>15}")
print("-" * 50)
for r in throughput_results:
    print(f"{r['batch_size']:<10} {r['throughput_ips']:>12.1f} ips {r['latency_per_batch_ms']:>12.2f}ms")

## 7️⃣ GPU Memory Usage

In [ ]:
# GPU Memory snapshot
gpu_memory = None
if torch.cuda.is_available():
    print("=" * 60)
    print("GPU MEMORY USAGE")
    print("=" * 60)
    
    torch.cuda.reset_peak_memory_stats()
    
    dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    model(dummy, verbose=False)
    torch.cuda.synchronize()
    
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    
    print(f"  Allocated:  {allocated:.2f} GB")
    print(f"  Reserved:   {reserved:.2f} GB")
    print(f"  Peak:       {peak:.2f} GB")
    
    gpu_memory = {
        'allocated_gb': round(allocated, 2),
        'reserved_gb': round(reserved, 2),
        'peak_gb': round(peak, 2)
    }

## 8️⃣ Generate Summary Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Resolution vs FPS
ax1 = axes[0, 0]
res_labels = [f"{r['imgsz']}" for r in speed_results]
fps_vals = [r['fps'] for r in speed_results]
bars = ax1.bar(res_labels, fps_vals, color='steelblue', edgecolor='black')
ax1.set_xlabel('Resolution')
ax1.set_ylabel('FPS')
ax1.set_title('FPS vs Resolution (T4 GPU)', fontweight='bold')
for bar, v in zip(bars, fps_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 5, f'{v:.0f}', ha='center', fontweight='bold')

# Plot 2: Batch Throughput
ax2 = axes[0, 1]
if throughput_results:
    batch_labels = [str(r['batch_size']) for r in throughput_results]
    throughput_vals = [r['throughput_ips'] for r in throughput_results]
    ax2.bar(batch_labels, throughput_vals, color='forestgreen', edgecolor='black')
    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('Images/sec')
    ax2.set_title('Throughput vs Batch Size', fontweight='bold')

# Plot 3: Accuracy Comparison
ax3 = axes[1, 0]
metrics = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
val_vals = [val_results.box.map50, val_results.box.map, val_results.box.mp, val_results.box.mr]
test_vals = [test_results.box.map50, test_results.box.map, test_results.box.mp, test_results.box.mr]

x = np.arange(len(metrics))
width = 0.35
ax3.bar(x - width/2, val_vals, width, label='Validation', color='coral', edgecolor='black')
ax3.bar(x + width/2, test_vals, width, label='Test', color='teal', edgecolor='black')
ax3.set_ylabel('Score')
ax3.set_title('Model Accuracy', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(metrics, rotation=15)
ax3.legend()
ax3.set_ylim(0, 1.1)

# Plot 4: CPU vs GPU Comparison
ax4 = axes[1, 1]
cpu_fps = [14.0, 7.4]  # Your CPU benchmarks
gpu_fps = [speed_results[0]['fps'], speed_results[1]['fps']]  # 416 and 640
labels = ['416x416', '640x640']

x = np.arange(len(labels))
ax4.bar(x - width/2, cpu_fps, width, label='CPU (i5-5250U)', color='#ff7f0e', edgecolor='black')
ax4.bar(x + width/2, gpu_fps, width, label='GPU (T4)', color='#1f77b4', edgecolor='black')
ax4.set_ylabel('FPS')
ax4.set_title('CPU vs GPU Speed Comparison', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(labels)
ax4.legend()

# Add speedup annotations
for i, (c, g) in enumerate(zip(cpu_fps, gpu_fps)):
    speedup = g / c
    ax4.text(i + width/2, g + 10, f'{speedup:.1f}x', ha='center', fontweight='bold', color='green')

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/gpu_benchmark_summary.png', dpi=150)
plt.show()
print(f"\n✅ Saved: {REPORTS_DIR}/gpu_benchmark_summary.png")

## 9️⃣ Save All Results

In [ ]:
# Compile all results
full_results = {
    'timestamp': datetime.now().isoformat(),
    'hardware': {
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
        'cuda_version': torch.version.cuda,
        'pytorch_version': torch.__version__,
        'gpu_memory': gpu_memory,
    },
    'model': {
        'path': str(MODEL_PATH),
        'num_classes': len(model.names),
    },
    'accuracy': {
        'validation': {
            'mAP50': round(val_results.box.map50, 4),
            'mAP50_95': round(val_results.box.map, 4),
            'precision': round(val_results.box.mp, 4),
            'recall': round(val_results.box.mr, 4),
        },
        'test': {
            'mAP50': round(test_results.box.map50, 4),
            'mAP50_95': round(test_results.box.map, 4),
            'precision': round(test_results.box.mp, 4),
            'recall': round(test_results.box.mr, 4),
        },
    },
    'speed': {
        'static_benchmark': speed_results,
        'video_benchmark': {
            'avg_fps': video_results['avg_fps'] if video_results else None,
            'max_fps': video_results['max_fps'] if video_results else None,
            'min_fps': video_results['min_fps'] if video_results else None,
            'std_fps': video_results['std_fps'] if video_results else None,
        } if video_results else None,
        'batch_throughput': throughput_results,
    }
}

# Save JSON
output_path = f'{REPORTS_DIR}/gpu_benchmark_full.json'
with open(output_path, 'w') as f:
    json.dump(full_results, f, indent=2)

print("\n" + "=" * 70)
print("📊 FINAL RESULTS SAVED")
print("=" * 70)
print(f"\n{json.dumps(full_results, indent=2)}")
print(f"\n✅ Saved to: {output_path}")

## 📋 Summary for README

Copy these values to update your README:

In [ ]:
print("\n" + "=" * 70)
print("📋 COPY THIS TO YOUR README")
print("=" * 70)

print(f"""
## GPU Benchmarks (NVIDIA T4)

| Metric | Value |
|--------|-------|
| **Accuracy (Test Set)** | |
| mAP@0.5 | {test_results.box.map50*100:.1f}% |
| mAP@0.5:0.95 | {test_results.box.map*100:.1f}% |
| Precision | {test_results.box.mp*100:.1f}% |
| Recall | {test_results.box.mr*100:.1f}% |
| **Speed** | |
| FPS (416x416) | {speed_results[0]['fps']} |
| FPS (640x640) | {speed_results[1]['fps']} |
| FPS (1280x1280) | {speed_results[2]['fps']} |
| Peak Throughput | {max([r['throughput_ips'] for r in throughput_results]):.0f} images/sec |

### CPU vs GPU Comparison

| Resolution | CPU (i5-5250U) | GPU (T4) | Speedup |
|------------|----------------|----------|--------|
| 416x416 | 14.0 FPS | {speed_results[0]['fps']} FPS | **{speed_results[0]['fps']/14:.1f}x** |
| 640x640 | 7.4 FPS | {speed_results[1]['fps']} FPS | **{speed_results[1]['fps']/7.4:.1f}x** |
""")